In [1]:
import sqlite3
import pandas as pd

In [38]:
pd.options.display.float_format = '{:.2f}'.format

In [2]:
# database file path
db_path = "D:/data-analyst-skill-gap/db/jobs.db"

In [3]:
# establish connection to database
conn = sqlite3.connect(db_path)

In [23]:
query = """SELECT skill, 
                  count(job_id) AS no_of_occurances 
            FROM job_skills 
            GROUP BY skill 
            ORDER BY count(job_id) DESC;"""
df1 = pd.read_sql_query(query,conn);
print(df1)

               skill  no_of_occurances
0                SQL                66
1           Power BI                44
2             Python                35
3              Excel                33
4            Tableau                26
5         Statistics                22
6                ETL                15
7   Machine Learning                 6
8                  R                 5
9              Spark                 4
10           Alteryx                 4
11               VBA                 3
12            Looker                 3
13               AWS                 3
14               SAS                 2
15         Big Query                 2
16             Azure                 2


In [24]:
# Sum of total occurances
print("Total occurances: ",df1['no_of_occurances'].sum())

Total occurances:  275


In [27]:
query = 'SELECT count(*) FROM job_skills;'
print(pd.read_sql_query(query,conn))

   count(*)
0       275


In [28]:
query = """SELECT job_id, 
                  COUNT(skill) AS num_skills 
            FROM job_skills 
            GROUP BY job_id 
            ORDER BY num_skills DESC;"""
df2 = pd.read_sql_query(query,conn);
print("Number of skills eteccteed for each job:\n",df2)

Number of skills eteccteed for each job:
          job_id  num_skills
0    5777073838           8
1    5777073828           8
2    5828954610           6
3    5812673893           6
4    5832798187           5
..          ...         ...
112  4686704927           1
113  2572832371           1
114  2449398766           1
115  2403881225           1
116  1993882092           1

[117 rows x 2 columns]


In [32]:
# % of job postings for which skills were detected
total_num_job_extracted = 450
jobs_with_skills = (df2.shape[0]/total_num_job_extracted)*100
print('Percentage of job postings where skilles were captured are: ',jobs_with_skills,'%')

Percentage of job postings where skilles were captured are:  26.0 %


In [41]:
# For each skill, what's the average salary of postings requiring it?
# Skill-wise average salary is correlational, not causal/attributive -  Since most 
# postings require multiple skills simultaneously, a jobs full salary contributes 
# to the average of *every* skill it lists — the number reflects average salary of 
# postings mentioning this skill, not the marginal value of this skill alone.
# See notebook for detailed reasoning.
query = """ SELECT  j.country, js.skill, 
                    AVG(j.avg_salary) as avg_sal 
            FROM jobs j 
            INNER JOIN job_skills js 
            ON j.job_id = js.job_id 
            GROUP BY j.country, js.skill 
            ORDER BY AVG(j.avg_salary) DESC;"""
print(pd.read_sql_query(query,conn))

           country             skill    avg_sal
0            India         Big Query 1350000.00
1            India             Spark 1025000.00
2            India  Machine Learning 1000000.00
3            India            Python  779166.67
4            India           Alteryx  750000.00
5            India               SQL  739722.22
6            India             Azure  700000.00
7            India             Excel  668125.00
8            India           Tableau  550833.33
9            India          Power BI  454500.00
10           India        Statistics  368125.00
11   United States           Alteryx  159403.75
12   United States             Spark  141000.00
13   United States               ETL  134360.39
14   United States           Tableau  130016.95
15   United States             Azure  127800.83
16   United States               AWS  121141.79
17   United States          Power BI  111081.13
18   United States        Statistics  106551.49
19   United States            Python  10

In [49]:
query = """ SELECT j.country, js.skill, AVG(j.avg_salary) AS avg_salary, COUNT(js.job_id) AS skill_occurance_count
            FROM jobs j
            INNER JOIN job_skills js
            ON j.job_id = js.job_id
            GROUP BY j.country, js.skill
            ORDER BY AVG(j.avg_salary) DESC, COUNT(js.job_id) DESC;"""
print(pd.read_sql_query(query,conn))     

           country             skill  avg_salary  skill_occurance_count
0            India         Big Query  1350000.00                      2
1            India             Spark  1025000.00                      3
2            India  Machine Learning  1000000.00                      3
3            India            Python   779166.67                     25
4            India           Alteryx   750000.00                      2
5            India               SQL   739722.22                     38
6            India             Azure   700000.00                      1
7            India             Excel   668125.00                     22
8            India           Tableau   550833.33                     20
9            India          Power BI   454500.00                     22
10           India        Statistics   368125.00                     14
11   United States           Alteryx   159403.75                      2
12   United States             Spark   141000.00                

In [57]:
query = """ WITH skill_sal_corr AS 
            (
            SELECT j.country, js.skill, AVG(j.avg_salary) as avg_salary, count(j.job_id) as skill_occurance_count,
            DENSE_RANK() OVER(PARTITION BY j.country ORDER BY count(j.job_id) DESC) as rank
            FROM jobs j
            INNER JOIN job_skills js
            ON j.job_id = js.job_id
            GROUP BY j.country,js.skill
            )
            SELECT *
            FROM skill_sal_corr
            WHERE rank = 1;
            """
print(pd.read_sql_query(query,conn))

          country     skill  avg_salary  skill_occurance_count  rank
0           India       SQL   739722.22                     38     1
1  United Kingdom  Power BI    55644.54                     11     1
2   United States       SQL    97009.39                     21     1


In [63]:
query = """ SELECT country, count(country) as count 
            FROM jobs
            WHERE avg_salary is NULL
            GROUP BY country; """
print(pd.read_sql_query(query,conn))

  country  count
0   India    109
